#### mysql과 python 연동에서의 유의할 점 
- 서버의 정보(주소, port, 아이디, 비밀번호, 데이터베이스명)를 코드에 그대로 입력 한 뒤 github에 코드를 업로드 
    - 보안상 취약점 
        - 내 집 주소와 현관문의 비밀번호를 알려주는 꼴
        - github에 해당 데이터들을 업로드를 하면 -> 확인하고 파일에 대해서 락 
        - openkey -> openapi와 같은 곳에서 비밀번호를 제공 
        - 이런 민감한 정보들은 숨겨서 관리 -> python-dotenv 라이브러리 설치하여 사용

In [ ]:
# !pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# 설치 할때 사용하는 라이브러리의 이름과 import 할때의 이름이 다른 경우들이 종종 발생 
from dotenv import load_dotenv
import pymysql 
import os 
import pandas as pd

In [2]:
# os -> 환경변수 (python 환경에서 사용하는 변수들에 접근하기 위해 사용)
# load_dotenv -> 환경변수에 등록 (.env 파일의 내용을 환경 변수에 등록(임시 등록))
load_dotenv()

True

In [3]:
# 등록된 환경변수에 접근 
# 가져온다(get) + 환경(env) -> getenv(변수명) 
os.getenv('pwd')

'1234'

In [6]:
# DB와 연결 
_db = pymysql.connect(
    host = os.getenv('host'),  
    # getenv() 함수의 결과는 문자 -> port는 숫자를 사용 -> 문자를 숫자로 변환
    port = int( os.getenv('port') ), 
    user = os.getenv('user'), 
    password = os.getenv('pwd'), 
    db = os.getenv('db_name')
)

In [ ]:
# Cursor 생성 (select의 결과값을 Dict 형태로 받아온다.)
cursor = _db.cursor( pymysql.cursors.DictCursor )

In [ ]:
# 새로운 환경에서 작업이 시작이 될때 모든 테이블의 구조를 다음과 같은 쿼리문을 이용하여 생성 

# 테이블 생성 -> 기존의 테이블이 존재한다면 테이블 생성x -> 없으면 생성
create_table = """
    CREATE TABLE IF NOT EXISTS
    `user_info` 
    (
        `id` VARCHAR(32) PRIMARY KEY, 
        `password` VARCHAR(32) NOT NULL, 
        `name` VARCHAR(32), 
        `age` INT
    )
"""
# DDL 구문은 생성 -> cursor에 질의를 보낸다 -> 테이블을 생성하는 과정은 확인 절차 없이 DB server에 바로 등록 
# cursor를 통해서 sql 쿼리문 실행 -> execute() 함수를 이용
cursor.execute(create_table)

0

In [ ]:
# 테이블에 데이터를 대입 -> insert 구문 
signup_query = """
    INSERT INTO `user_info` 
    VALUES ("test", "1234", "kim", 30)
"""

cursor.execute(signup_query)

1

In [ ]:
user_list_query = """
    SELECT * FROM `user_info`
"""

cursor.execute(user_list_query)


1

In [14]:
# 결과 값을 불러오는 함수 -> fetchall()
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30}]

In [15]:
# 결과를 DB와의 동기화(확인 후 확정)
# _db에서 확정 작업(commit())
_db.commit()

In [18]:
# query 문에 들어오는 데이터가 매번 바뀔때마다 query문을 작성하는건 불필요한 작업 
signup_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
# %s는 들어오는 데이터들의 위치를 지정 
input_id = input("아이디를 입력하시오")
input_pass = input("비밀번호를 입력하시오")
input_name = input('이름을 입력하시오')
input_age = input('나이를 입력하시오')

# execute(query, [datas]) 함수에 리스트의 각각의 원소들이 %s 위치에 순서대로 대입 
cursor.execute(signup_query, [ input_id, input_pass, input_name, input_age ])

1

In [19]:
cursor.execute(user_list_query)

2

In [20]:
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test2', 'password': 'park', 'name': 'park', 'age': 20}]

#### DB 연동 class 선언 
- 생성자 함수 
    - 서버의 정보를 입력한다. 
    - class가 생성이 될때마다 다른 DB server의 정보를 담을수 있다. 
    - 2개의 객체를 생성하여 다른 DB server와의 연동 
- 함수 생성 -> 2개 생성 
    - query문 실행 함수 ( 매개변수 : query문(필수 입력), *datas(query에 %s가 존재하지 않으면 필요x) )
        - 서버와의 연결
        - cursor를 생성
        - CUD ( insert, update, delete )
            - query문이 작성 
            - execute() 함수를 이용하여 cursor에 질의를 보낸다 
                - cursor의 테이블에서 데이터의 변화 
        - R ( select )
            - query문이 작성 
            - execute() 함수를 이용하여 cursor에 질의를 보낸다. 
            - cursor에서 데이터를 가져온다 (fetchall())
        - query문의 시작이 select라면 fetchall()함수를 사용하여 데이터를 리턴
    - DataBase에 변화를 주는 함수 
        - DB server에서 commit()를 이용해서 데이터 확정 
        - DB server와의 연결을 종료 (close())
        

In [ ]:
# class 선언 
class MyDB:
    # 생성자 함수 (서버의 정보를 받아오기 위해 사용)
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db

    # 데이터베이스에 변화를 주는 함수 
    def commit(self):
        try:
            # DataBase에 데이터를 확정 (동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료 ( 중요한 부분 )
            self.db_server.close()
        except:
            # 문제가 발생하는 이유는? -> 서버와의 연결이 되지 않은 경우 (self.db_server라는 변수에 데이터가 없거나 아예 존재하지 않는 경우)
            print( "데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요" )
    
    def sql_query(self, query, *datas):
        # query : sql query문이 입력이되는 매개변수 
        # datas : query문에서 사용이 될 데이터의 목록

        # DB_server와의 연결 
        self.db_server = pymysql.connect(
            host = self.host, 
            port = self.port, 
            user = self.user, 
            password = self.password, 
            db = self.db
        )
        # cursor 생성 
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        # self.변수명 // 변수명의 차이는?
            # self.변수 -> 독립적으로 객체 안에 저장이 되는 변수 ( 함수 호출 후에도 데이터가 존재 )
            # 변수 -> 함수 호출시 생성이 되고 함수가 종료가 되면 휘발성으로 사라짐